In [1]:
from bs4 import XMLParsedAsHTMLWarning
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

import sys
sys.path.append("..")  # src 모듈을 import 하기 위해 상위 폴더를 경로에 추가

from src.edgar_client import fetch_filing_html
from src.parser import find_schedule_of_investments_tables, _row_cells, parse_filing_html

url = "https://www.sec.gov/Archives/edgar/data/1736035/000173603526000010/bxsl-20260331.htm"
html = fetch_filing_html(url)

tables = find_schedule_of_investments_tables(html)
print(f"찾은 테이블 개수: {len(tables)}")

찾은 테이블 개수: 136


In [2]:
# from src.parser import find_schedule_of_investments_tables, _table_as_of_date
# from src.edgar_client import extract_period_end_date

# tables = find_schedule_of_investments_tables(html)
# for i, table in enumerate(tables):
#     print(i, _table_as_of_date(table))

In [3]:
# from src.pipeline import build_time_series, save_to_sqlite
# from config import BXSL_CIK

# parsed, unmatched = build_time_series(BXSL_CIK, form_types=["10-Q", "10-K"], limit=8)
# print(parsed.shape, unmatched.shape)

# save_to_sqlite(parsed, unmatched)

In [4]:
# from src.edgar_client import get_recent_filings, build_document_url, fetch_filing_html
# from src.parser import find_schedule_of_investments_tables, _table_as_of_date, _row_cells
# from config import BXSL_CIK

# # 2024-06-30 filing만 다시 열어서 Aevex 관련 원본 행 확인
# filings = get_recent_filings(BXSL_CIK, form_types=["10-Q"], limit=8)
# target = [f for f in filings if f["filing_date"] == "2024-08-07"][0]
# url = build_document_url(target["cik"], target["accession_number"], target["primary_document"])
# html_2024q2 = fetch_filing_html(url)

# tables_2024q2 = find_schedule_of_investments_tables(html_2024q2)
# current_2024q2 = [t for t in tables_2024q2 if _table_as_of_date(t) == "2024-06-30"]

# for table in current_2024q2:
#     for tr in table.find_all("tr"):
#         cells = _row_cells(tr)
#         if any("Aevex" in c for c in cells):
#             print(cells)

In [5]:
# parsed[parsed["investment_name"] == "Aevex Holdings, LLC"][
#     ["investment_name", "fair_value", "period_end_date", "acquisition_date"]
# ].sort_values("period_end_date")

In [6]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import sqlite3
import pandas as pd

from src.metrics import clean_numeric_columns, detect_events
from src.normalizer import normalize_entity_name

conn = sqlite3.connect("../data/bdc_tracker.db")
positions = pd.read_sql("SELECT * FROM positions", conn)
conn.close()

positions_clean = clean_numeric_columns(positions)

In [7]:
from src.normalizer import normalize_entity_name

print(normalize_entity_name("BP Alpha Holdings, L.P."))
print(normalize_entity_name("BP Alpha Holdings, LP"))

bp alpha holdings
bp alpha holdings


In [8]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

box_events = events[events["investment_name"].str.contains("Box Co-Invest", case=False, na=False)]
print(box_events[["investment_name", "event_type", "period_end_date", "fair_value"]])

event_type
new_entry    643
markup       219
exit         141
markdown     102
Name: count, dtype: int64
                                       investment_name event_type  \
120  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
204  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
466  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
496  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
608  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   

    period_end_date  fair_value  
120      2024-09-30       155.0  
204      2024-09-30       102.0  
466      2024-12-31         0.0  
496      2024-12-31        16.0  
608      2025-03-31         0.0  


In [9]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

box_events = events[events["investment_name"].str.contains("Box Co-Invest", case=False, na=False)]
print(box_events[["investment_name", "event_type", "period_end_date", "fair_value"]])

event_type
new_entry    643
markup       219
exit         141
markdown     102
Name: count, dtype: int64
                                       investment_name event_type  \
120  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
204  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
466  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
496  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
608  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   

    period_end_date  fair_value  
120      2024-09-30       155.0  
204      2024-09-30       102.0  
466      2024-12-31         0.0  
496      2024-12-31        16.0  
608      2025-03-31         0.0  


In [10]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

event_type
new_entry    643
markup       219
exit         141
markdown     102
Name: count, dtype: int64


In [11]:
# from src.ai_classifier import classify_unmatched_rows

# sample = unmatched.head(15)
# classified_sample = classify_unmatched_rows(sample)
# classified_sample

In [12]:
# classified_sample = classify_unmatched_rows(unmatched)
# classified_sample[classified_sample["investment_type"] == "fx_forward"][
#     ["counterparty", "currency_purchased_amount", "currency_purchased_code",
#      "currency_sold_amount", "currency_sold_code", "settlement_date", "unrealized_gain_loss"]
# ]

In [13]:
conn = sqlite3.connect("../data/bdc_tracker.db")
positions = pd.read_sql("SELECT * FROM positions", conn)
conn.close()

bf_in_parsed = positions[
    positions["investment_name"].astype(str).str.contains(
        "BlackRock|Fidelity|State Street", case=False, na=False
    )
]
print(len(bf_in_parsed))
bf_in_parsed[["investment_name", "fair_value", "period_end_date"]].head(10)

15


,investment_name,fair_value,period_end_date
693,BlackRock ICS US Treasury Fund,166,2026-03-31
694,Fidelity Investments Money Market Treasury Por...,2,2026-03-31
695,State Street Institutional U.S. Government Mon...,"7,158",2026-03-31
696,State Street Institutional U.S. Government Mon...,"38,156",2026-03-31
1687,State Street Institutional U.S. Government Mon...,"6,807",2025-12-31
1688,State Street Institutional U.S. Government Mon...,"17,168",2025-12-31
1689,BlackRock ICS US Treasury Fund,"3,167",2025-12-31
2686,State Street Institutional U.S. Government Mon...,440,2025-09-30
2687,BlackRock ICS US Treasury Fund,103,2025-09-30
3621,State Street Institutional U.S. Government Mon...,"20,521",2025-06-30


In [14]:
from src.pipeline import build_time_series, save_to_sqlite
from config import BXSL_CIK

parsed, unmatched = build_time_series(BXSL_CIK, form_types=["10-Q", "10-K"], limit=8)
print(parsed.shape, unmatched.shape)

save_to_sqlite(parsed, unmatched)

Fetching 2026-05-07 (10-Q), period end: 2026-03-31...
[parser] 42 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2026-02-25 (10-K), period end: 2025-12-31...
[parser] 37 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-11-10 (10-Q), period end: 2025-09-30...
[parser] 35 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-08-06 (10-Q), period end: 2025-06-30...
[parser] 34 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-05-07 (10-Q), period end: 2025-03-31...
[parser] 24 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2025-02-26 (10-K), period end: 2024-12-31...
[parser] 18 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
Fetching 2024-11-12 (10-Q), 

In [ ]:
%load_ext autoreload
from src.parser import parse_filing_html

parsed_test, unmatched_test = parse_filing_html(html, target_period_end="2026-03-31")

swap_check = unmatched_test[
    unmatched_test["raw_tokens"].astype(str).str.contains("SMBC|April 2028 Notes", case=False, na=False)
]
print(swap_check[["section_heading", "raw_tokens"]].to_string())

fx_check = unmatched_test[
    unmatched_test["raw_tokens"].astype(str).str.contains("Wells Fargo", case=False, na=False)
]
print(fx_check["section_heading"].value_counts())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[parser] 42 rows didn't match the standard debt-row pattern — check the 'raw_tokens' column for those rows.
                         section_heading                                                                                                      raw_tokens
1029  Foreign Currency Forward Contracts  [SMBC Capital Markets, Inc., November 2027 Notes, 5.88%, SOFR +, 1.38%, 11/15/2027, 400,000, 11,205, —, 2,954]
1030  Foreign Currency Forward Contracts           [Wells Fargo Bank, N.A., April 2028 Notes, 5.35%, SOFR +, 1.65%, 4/13/2028, 400,000, 5,343, —, 2,855]
1031  Foreign Currency Forward Contracts         [Wells Fargo Bank, N.A., April 2028 Notes, 5.35%, SOFR +, 1.39%, 4/13/2028, 300,000, 1,834, —, (1,969)]
